# GP_ELITE — fifteen minutes, five steps

**You have a table of measurements and you want the formula, not a prediction.**
This notebook starts from nothing and ends on *your* data.

Nothing to install on your machine: everything runs in the browser.
Run the cells in order with **Shift + Enter**.

*[🇫🇷 Version française](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/quickstart.fr.ipynb)*

---


## 1. Install

Three dependencies, no compiler, no second language runtime.
A few seconds — and that is the point of everything below.


In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'ready.')


## 2. Your first law: Kepler, from eight points

Distance to the Sun and orbital period of the eight planets. Nothing else.

Kepler took years to establish this relationship. Let us see what a machine
makes of it with eight rows of data.


In [ ]:
import numpy as np
from gp_elite import symbolic_regression

distance = np.array([0.387, 0.723, 1.000, 1.524, 5.203, 9.537, 19.191, 30.069])
period   = np.array([0.241, 0.615, 1.000, 1.881, 11.862, 29.457, 84.011, 164.79])

result = symbolic_regression(
    distance.reshape(-1, 1), period,
    feature_names=['distance'],
    operators='physical', generations=30, speed='fast', seed=0,
)
print(result.expression)


You should read something like `164.78 * (distance * sqrt(distance))`,
that is **distance^1.5** — Kepler's third law.

The coefficient is only a unit factor. The *shape* is the law.


## 3. Physical units: what the others do not do

Here, a spring experiment. We measure an elongation in metres and a force in
newtons, and we are looking for Hooke's law `F = k·x`.

The stiffness `k` **is not in the data** — it is an unknown physical constant,
and it carries units. We ask the engine to deduce it.


In [ ]:
from gp_elite import GPEliteRegressor

rng = np.random.RandomState(0)
elongation = rng.uniform(0.01, 0.10, 150).reshape(-1, 1)   # metres
force = 250.0 * elongation[:, 0]                            # newtons

est = GPEliteRegressor(
    units=['m'],              # unit of the input column
    target_units='N',         # unit of the target
    unknown_constant=True,    # the missing constant is to be deduced
    generations=25, speed='fast', random_state=0,
)
est.fit(elongation, force)

print('deduced units :', est.constant_units_string())
print('deduced value :', round(est.constant_value_, 4))


`[kg / s^2]` and `250.0` — exactly newtons per metre, and the exact stiffness
of the spring.

The engine did not merely find the shape of the law: it reported **the units
and the value of a physical quantity absent from the data**.

Without `units=`, this problem is out of reach — no dimensionless constant can
relate metres to newtons.


---

## 4. Real data: what measurements from 1933 say

Everything above was clean. This is not.

In the early 1930s Johann Nikuradse glued sand grains of a known size inside pipes and
measured the friction of turbulent flow: **362 real measurements**, six roughness ratios,
Reynolds numbers from 4,300 to 1,000,000. The data is still in use today — the full
dependence of friction on Reynolds number and roughness is regarded as an open problem.

The engine gets the raw numbers and nothing else: the roughness ratio `r_k`, the base-10
log of the Reynolds number, and the friction as `log10(100*lambda)`.

The bar to clear is the textbook **Prandtl–von Kármán** relation, which in this target
space reads `2 - 2*log10(2*log10(r/k) + 1.74)` — with no fitted parameters at all.

In [ ]:
import pandas as pd

URL = ("https://github.com/EpistasisLab/pmlb/raw/master/datasets/"
       "nikuradse_1/nikuradse_1.tsv.gz")
df = pd.read_csv(URL, sep="\t", compression="gzip")

Xn = df[["r_k", "log_Re"]].to_numpy(float)
yn = df["target"].to_numpy(float)

def prandtl_von_karman(r_k):
    return 2.0 - 2.0 * np.log10(2.0 * np.log10(r_k) + 1.74)

print(f"{len(yn)} measurements, roughness ratios {sorted(df.r_k.unique())}")
r2_pvk = 1 - np.mean((prandtl_von_karman(Xn[:, 0]) - yn)**2) / np.var(yn)
print(f"Prandtl-von Karman, zero fitted parameters: R2 = {r2_pvk:.4f}")

Now the search. **This cell takes a few minutes**: real data has no exact solution to stop
at, so the engine spends its whole budget.

Under the full protocol (`generations=30, restarts=4`) GP_ELITE returns a seven-node
expression, `1.32 - 0.56*log(log(r_k))`, scoring R² 0.9455 against the reference law's
0.9456 — a dead heat — and tracking it with a correlation of 0.9992 across the six
roughness ratios. The shorter run below may land elsewhere; the table it prints tells you
honestly what it found, whatever that is.

In [ ]:
import time

t0 = time.time()
m = symbolic_regression(Xn, yn, feature_names=['r_k', 'log_Re'],
                        operators='physical', normalize='none',
                        generations=30, speed='fast', restarts=2, seed=0)
print(f'searched for {time.time()-t0:.0f} s\n')

def r2(e):
    return 1 - np.mean((e.predict(Xn) - yn)**2) / np.var(yn)

front = sorted({(int(e.size), round(r2(e), 6)): e
                for e in list(m.pareto or []) + [m]}.items())
print('size   R2      expression')
for (s, r), e in front:
    print(f'{s:>4}  {r:>6.4f}  {e.expression[:78]}')

good = [(k, e) for k, e in front if k[1] > 0.90]
if good:
    (s, r), e = good[0]
    six = np.array(sorted(df.r_k.unique()))
    grid = np.column_stack([six, np.full(len(six), Xn[:, 1].mean())])
    pred, theory = e.predict(grid), prandtl_von_karman(six)
    print(f'\nmost compact expression above R2=0.90 ({s} nodes):\n  {e.expression}')
    print(f"\n{'r/k':>7}{'Prandtl-von Karman':>21}{'GP_ELITE':>12}")
    for a, b, c in zip(six, theory, pred):
        print(f'{a:>7.1f}{b:>21.4f}{c:>12.4f}')
    print(f'\ncorrelation with the textbook law: '
          f'{np.corrcoef(theory, pred)[0, 1]:.4f}')
else:
    print('\nnothing compact above R2=0.90 in this short run - see '
          'benchmarks/real_nikuradse.py for the full protocol')

**What this shows, and what it does not.** GP_ELITE *rediscovers* a known law here; it does
not beat it. On a roughness ratio removed from training entirely, the textbook relation
predicts the level to within 0.04 standard deviations of the curve, against 0.25 for the
recovered form — a systematic 2.6% approximation gap.

That is the honest summary, and it is the point: from raw measurements the engine found the
right *family* of functions, with constants close enough to be useful and far enough to be
visible. Full protocol, held-out-roughness test and telemetry live in
`benchmarks/real_nikuradse.py`.

## 5. Your turn

Load a CSV: one target column, one or more input columns.
A hundred to a few thousand rows, up to about ten variables.


In [ ]:
import pandas as pd, io

try:
    from google.colab import files
    upload = files.upload()
    name = list(upload)[0]
    df = pd.read_csv(io.BytesIO(upload[name]))
except ImportError:
    df = pd.read_csv('my_data.csv')   # outside Colab: your path here

print(df.shape, 'rows x columns')
df.head()


In [ ]:
# Adapt these two lines to your columns.
TARGET   = df.columns[-1]
INPUTS   = [c for c in df.columns if c != TARGET]

X = df[INPUTS].to_numpy(dtype=float)
y = df[TARGET].to_numpy(dtype=float)

res = symbolic_regression(X, y, feature_names=INPUTS,
                          operators='physical', generations=40,
                          speed='fast', seed=0)
print(res.expression)
print('R² on data never seen :', res.r2_validation)


> **If one of your columns is an angle**, or feeds an exponential or a
> logarithm, add `normalize='none'`. Measured on the trigonometric equations
> of the Feynman benchmark, that single argument raises exact recoveries from
> 3 to 19 out of 50 runs, with models half the size.

---

## It did not work?

GP_ELITE is tuned on published benchmarks — clean, noise-free, well-scaled.
Your measurements are none of those things, and that gap is where the engine
most needs work.

A failure on your data is **useful information, not user error**.
[Open an issue](https://github.com/ariel95500-create/gp-elite/issues/new/choose)
with the shape of your data and what you got back — no need to share the data,
no need to know why. English or French.

## Going further

- [README](https://github.com/ariel95500-create/gp-elite#readme) — robust mode,
  Pareto front, extrapolation, dimensional audit
- `gp-elite` on the command line: an interactive console, no Python required
- [CHANGELOG](https://github.com/ariel95500-create/gp-elite/blob/main/CHANGELOG.md)
  — every claim in the README is backed by a reproducible measurement
